# 3.4 Survival Analysis

**Core series, class 9.** Among people with type 2 diabetes, **how long until chronic kidney disease (CKD)**, and who gets it sooner? In 1.2 we gathered the dates: time zero (the first diabetes date), the event (the first CKD date after it), and censoring (the end of each person's records). Now we analyse them with **Kaplan–Meier curves** and a **Cox model**.

### By the end of this notebook you'll be able to

1. Build **time-to-event** data: follow-up time and an event indicator, with censoring
2. Estimate and plot **Kaplan–Meier** curves with `survival` and **ggplot2**, and compare groups with a **log-rank test**
3. Fit a **Cox proportional-hazards** model, and read **hazard ratios**
4. Check the **proportional-hazards** assumption
5. Explain the main pitfalls of survival analysis in health records

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | After a type 2 diabetes diagnosis, how quickly do people develop CKD, and is it faster with higher HbA1c? |
| 🧪 **The analysis** | **Kaplan–Meier** curves by HbA1c group, a **log-rank** test, then a **Cox** model adjusted for age at diagnosis, sex, and BMI |
| 🎯 **Why** | People are followed for different lengths of time, and many leave the records without CKD. Survival methods use **everyone's** follow-up, through **censoring**, instead of pretending they never got CKD |
| ✅ **What we conclude** | The share CKD-free over time, and hazard ratios: how much faster CKD comes with each characteristic |
| ⚠️ **What it can't tell us** | Cause. And our HbA1c and BMI are the **latest** values, often measured after time zero; records that start late make time zero uncertain |
| 🔗 **Connections** | Uses the dates from **1.2**, joined in **2.3**; builds on regression in **3.2** and **3.3** |

| | |
|---|---|
| 📂 **Uses** | `final_analysis.csv` (from **2.3**) |
| 💾 **Saves** | nothing |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Build the survival data | 15 |
| 3. Kaplan–Meier curves | 25 |
| 4. Cox model | 15 |
| 5. Check the assumption | 10 |
| 6. Discuss | 10 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 90 minutes |
| ⏱️ **Run time** | under **1 minute** to run every cell |
| 💰 **Data cost** | **$0**: no BigQuery queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.40** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** The number-at-risk table can contain small counts at long follow-up times: check before sharing. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

| Term | Means | In our study |
|---|---|---|
| **Time zero** | when each person's clock starts | first type 2 diabetes date |
| **Event** | what we're waiting for | first CKD date after time zero |
| **Censored** | the clock stopped **without** the event | the earlier of the last EHR visit and death |
| **Status** | 1 = event, 0 = censored | |
| **Survival curve** S(t) | the share still **event-free** at time t | the share CKD-free t years after diagnosis |
| **Hazard** | the event rate at a moment, among those still at risk | |
| **Hazard ratio (HR)** | the hazard in one group ÷ another. HR 1.5 = the event happens 50% faster | |

### Kaplan–Meier

The **Kaplan–Meier** estimate steps down at each event. People who are censored leave the **risk set** (the people still being followed) without making the curve step down, so their follow-up counts for as long as we saw them.

### Cox proportional-hazards model

The **Cox** model is to survival what logistic regression is to yes/no: it gives **hazard ratios** for several predictors at once, each adjusted for the others. It assumes each HR stays **constant over time** (**proportional hazards**), which we check in Part 5.

### Functions

| Function | Package | What it does |
|---|---|---|
| `Surv(time, status)` | survival | the outcome: follow-up time + event indicator |
| `survfit(Surv(…) ~ group)` | survival | Kaplan–Meier curves |
| `survdiff(Surv(…) ~ group)` | survival | the **log-rank test**: do the curves differ? |
| `coxph(Surv(…) ~ x1 + x2)` | survival | the Cox model |
| `cox.zph(model)` | survival | tests the proportional-hazards assumption |
| `broom::tidy()` | broom | turns `survfit` and `coxph` results into tables, for ggplot2 |

## Part 2. Build the survival data

From the **cases** in our sample:

1. `end_of_records`: the earlier of the last EHR visit and death (`pmin(…, na.rm = TRUE)`)
2. **Exclude** people whose CKD was recorded **on or before** their first diabetes date (1.2)
3. `status`: 1 if CKD came after diabetes, else 0
4. `end_date`: the CKD date for events, the end of records for everyone else
5. `time_years`: from the first diabetes date to `end_date`; keep positive times only
6. Covariates: **age at diagnosis** (per 10 years), sex, BMI (per 5 units), and an **HbA1c group**: 7% or more vs below 7%, a common treatment target

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions
library(survival)
library(broom)

fa <- prepare_analysis_data()

surv_data <- fa %>%
  filter(t2d == "Yes") %>%
  mutate(end_of_records = pmin(last_ehr_date, death_date, na.rm = TRUE)) %>%
  filter(is.na(first_ckd_date) | first_ckd_date > first_t2d_date) %>%              # 2. CKD before diabetes: excluded
  mutate(status     = if_else(!is.na(first_ckd_date), 1, 0),                        # 3.
         end_date   = if_else(status == 1, first_ckd_date, end_of_records),         # 4.
         time_years = as.numeric(end_date - first_t2d_date) / 365.25,               # 5.
         age10_at_t2d = as.numeric(first_t2d_date - date_of_birth) / 365.25 / 10,  # 6.
         BMI5        = BMI / 5,
         hba1c_group = factor(if_else(HbA1c >= 7, "HbA1c 7% or more", "HbA1c below 7%"),
                              levels = c("HbA1c below 7%", "HbA1c 7% or more"))) %>%
  filter(!is.na(time_years), time_years > 0)

nrow(surv_data)
surv_data %>% count(status)
summary(surv_data$time_years)

**🔍 Read the output:** the number of cases in the analysis (fewer than 5,000: people with CKD before diabetes, or with no follow-up after it, were removed); how many had the event (`status` 1) and how many were censored (0); and the distribution of follow-up time.

## Part 3. Kaplan–Meier curves

### Everyone together

`survfit(Surv(time_years, status) ~ 1)`: the `~ 1` means one curve for everyone. `summary(…, times = …)` gives the estimate at chosen times.

In [ ]:
km_all <- survfit(Surv(time_years, status) ~ 1, data = surv_data)

km_all   # people, events, and the median time to CKD
summary(km_all, times = c(1, 3, 5, 10)) %>%
  with(tibble(years = time, at_risk = n.risk, ckd_free = round(surv, 3),
              conf.low = round(lower, 3), conf.high = round(upper, 3)))

**🔍 Read the output:**

- The first output: the number of people (`n`), `events`, and the **median** survival time (`NA` if fewer than half had the event, which is usual here: most people don't develop CKD within the follow-up).
- The table: the share still **CKD-free** 1, 3, 5, and 10 years after diagnosis, with 95% confidence intervals, and how many were still being followed (`at_risk`). Fewer people are at risk at longer times, so the estimates get less certain.

### By HbA1c group, drawn with ggplot2

`broom::tidy()` turns the curve into a table (one row per time point, per group), which ggplot2 can draw:

- `geom_step()`: Kaplan–Meier curves are **steps**, not smooth lines
- the dashed steps are the 95% confidence limits

In [ ]:
km_fit <- survfit(Surv(time_years, status) ~ hba1c_group, data = surv_data)
km_df  <- tidy(km_fit) %>%
  mutate(group = str_remove(strata, "hba1c_group=")) %>%
  bind_rows(tibble(time = 0, estimate = 1, conf.low = 1, conf.high = 1,      # start every curve at 100%
                   group = levels(surv_data$hba1c_group)))

ggplot(km_df, aes(time, estimate, colour = group)) +
  geom_step(linewidth = 0.9) +
  geom_step(aes(y = conf.low),  linetype = "dashed", alpha = 0.6) +
  geom_step(aes(y = conf.high), linetype = "dashed", alpha = 0.6) +
  coord_cartesian(xlim = c(0, 10), ylim = c(0.5, 1)) +
  scale_colour_manual(values = c("HbA1c below 7%" = "#3182bd", "HbA1c 7% or more" = "#d95f02")) +
  theme_bw() +
  labs(x = "Years since type 2 diabetes diagnosis", y = "Share CKD-free", colour = NULL,
       title = "Kaplan-Meier: time to CKD after type 2 diabetes")

survdiff(Surv(time_years, status) ~ hba1c_group, data = surv_data)

**🔍 Read the output:**

- **The curves:** each starts at 1 (everyone CKD-free) and steps down at each CKD diagnosis. A **lower** curve means CKD comes **sooner**. The y-axis starts at 0.5, to show the differences; most people stay CKD-free.
- **The log-rank test** (`survdiff`): `Observed` vs `Expected` events in each group if the curves were the same, and a p-value (`p=` at the bottom). A small p-value means the curves differ.
- People with no HbA1c in the data are left out of this comparison (they're in the "everyone" curve).

### Number at risk

A survival plot should come with the **number at risk**: how many people the curve is based on at each time.

In [ ]:
risk_table <- summary(km_fit, times = c(0, 2, 4, 6, 8, 10), extend = TRUE) %>%
  with(tibble(group = str_remove(strata, "hba1c_group="), years = time, n = n.risk)) %>%
  hide_small_counts() %>%
  pivot_wider(names_from = years, values_from = n, names_prefix = "year_")

risk_table

**🔍 Read the output:** the number still followed (CKD-free and not censored) at 0, 2, 4, … years. When these numbers get small, the curve's tail is unreliable. `hide_small_counts()` blanks any count of 1 to 20, so the table can be shared.

### ✍️ Your turn

Draw Kaplan–Meier curves by **sex at birth** instead, and run the log-rank test. Do men and women differ?

<details>
<summary><b>Show answer</b></summary>

```r
km_sex <- survfit(Surv(time_years, status) ~ sex_at_birth,
                  data = filter(surv_data, sex_at_birth %in% c("Female", "Male")))
tidy(km_sex) %>%
  ggplot(aes(time, estimate, colour = strata)) +
  geom_step() +
  coord_cartesian(xlim = c(0, 10), ylim = c(0.5, 1)) +
  theme_bw()

survdiff(Surv(time_years, status) ~ sex_at_birth,
         data = filter(surv_data, sex_at_birth %in% c("Female", "Male")))
```

Swap the variable after `~`. Keep only the main categories, so each curve has enough people.

</details>

In [ ]:
# Your code here

## Part 4. Cox model

The Cox model adjusts for several predictors at once. As in 3.2, it uses people with **every** variable; we report how many.

In [ ]:
cox_data <- surv_data %>%
  filter(sex_at_birth %in% c("Female", "Male")) %>%
  drop_na(age10_at_t2d, BMI5, hba1c_group)
nrow(cox_data)
cox_data %>% count(status)

cox_model <- coxph(Surv(time_years, status) ~ age10_at_t2d + sex_at_birth + BMI5 + hba1c_group,
                   data = cox_data)

tidy(cox_model, exponentiate = TRUE, conf.int = TRUE) %>%
  select(term, HR = estimate, conf.low, conf.high, p.value) %>%
  mutate(across(c(HR, conf.low, conf.high), ~ round(.x, 2)))

concordance(cox_model)$concordance

**🔍 Read the output:**

- `HR`: the **hazard ratio**, adjusted for the other predictors. HR 1.3 per 10 years of age means CKD comes 30% faster per decade older at diagnosis. A CI that doesn't include 1 means the link is unlikely to be chance.
- `hba1c_groupHbA1c 7% or more`: the HR for people with higher HbA1c compared with below 7%, at the same age, sex, and BMI.
- **Concordance** (0.5 to 1): how well the model ranks who gets CKD sooner. 0.5 is chance; 0.7 or more is useful.

## Part 5. Check the proportional-hazards assumption

`cox.zph()` tests, for each predictor, whether its HR **changes over time**. A **small p-value** (below 0.05) means the HR isn't constant for that predictor.

In [ ]:
cox.zph(cox_model)

**🔍 Read the output:** one row per predictor, plus `GLOBAL` for the whole model. If a predictor's p-value is small, its HR is an **average** over time, and the effect may be stronger early or late. Options: report that, split follow-up time into periods, or **stratify** by that variable (`strata(x)` in the formula). With large samples, small departures often show up as "significant", so also look at the size (`plot(cox.zph(cox_model))` shows how each effect changes over time).

## Part 6. Discuss

This analysis looks tidy, but health-record survival analyses have traps. Discuss each:

1. **Timing of covariates:** our HbA1c and BMI are each person's **latest** values, often measured **after** time zero, sometimes after CKD. A covariate measured after the start can be affected by the outcome. Better: the value **closest before** the diabetes date (a **baseline**).
2. **Late time zero (left truncation):** the "first diabetes date" in the records may be years after the real diagnosis, if records start late. How would that affect follow-up times?
3. **Competing risks:** people who **die** are censored, as if they could still get CKD later. With many deaths, Kaplan–Meier overestimates CKD risk. Methods for **competing risks** handle this.
4. **Who gets tested:** CKD is diagnosed with kidney function tests. People who see doctors more often are tested more, and diagnosed sooner (**surveillance bias**).
5. **The outcome definition:** one CKD code, or two? A lab-based definition (two low eGFR results 3 months apart) would be stricter.

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *Stop time must be > start time* / negative times | An end date before time zero | Keep `time_years > 0` (Part 2) |
| *there is no package called 'survival'* | Not installed | `aou_setup.R` installs it; rerun the setup cell |
| Curves that don't start at 1 | `tidy()` starts at the first event time | Add a time-0 row (as in Part 3) |
| *Loglik converged before variable …* | A predictor perfectly predicts the event in a group | Check counts by group; drop or merge levels |

## ✅ Key takeaways

- Survival analysis needs **time zero**, an **event**, and **censoring**
- **Kaplan–Meier** curves show the share event-free over time; the **log-rank test** compares them
- The **Cox model** gives **adjusted hazard ratios**; check **proportional hazards**
- In health records, watch **covariate timing**, **late time zero**, **competing risks**, and **who gets tested**

**This is the end of the core series.** The **omics** series (4.1–4.7) uses the same cohort with genetic, RNA, and protein data.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).